# Combining adversarial training with temporal structure

The original adversarial game compares observed data with samples produced from random inputs:

$$
\min_G\max_D V(D,G)
=\mathbb E_{X\sim p_{\mathrm{data}}}[\log D(X)]
+\mathbb E_{Z\sim p_Z}[\log(1-D(G(Z)))].
$$

Here, $G$ is the generator; $D(x)$ is the discriminator's assigned probability that $x$ came from the observed distribution; $X\sim p_{\mathrm{data}}$ is an observed sample; $Z\sim p_Z$ is a random input; $G(Z)$ is a generated sample. The discriminator maximizes the expression, while the generator minimizes it.

## Training latent sequences with three losses

The temporal model first maps an observed window $X$ into a latent sequence and then maps it back:

$$
H=E(X),
\qquad
\widetilde X=R(H),
\qquad
\widehat H=G(Z),
\qquad
\widehat X=R(\widehat H).
$$

Here, $E$ is the embedding map; $R$ is the recovery map; $H$ is the latent representation of an observed window; $G$ is an autoregressive generator; $Z$ is a sequence of independent Gaussian inputs; $\widehat H$ is a generated latent sequence; $\widetilde X$ is a reconstruction; $\widehat X$ is a generated feature window.

For a batch of $B$ windows, the reconstruction loss is

$$
L_R=\frac{1}{B}\sum_{i=1}^{B}\sum_{t=1}^{T}\lVert X_{i,t}-\widetilde X_{i,t}\rVert_2^2.
$$

Here, $i$ identifies a window; $t$ identifies a position within the window; $T$ is the window length; $\lVert\cdot\rVert_2$ is Euclidean length. The loss measures how much information is lost when a window is embedded and recovered.

The supervised temporal loss is

$$
L_S=\frac{1}{B}\sum_{i=1}^{B}\sum_{t=1}^{T}\lVert H_{i,t}-G_t(H_{i,1:t-1},Z_{i,t})\rVert_2^2.
$$

Here, $G_t(H_{i,1:t-1},Z_{i,t})$ is the latent vector produced at position $t$ when the generator receives the preceding observed latent history and the current random input. The loss compares the generated next latent vector with the latent vector obtained from the observed sequence.

The adversarial value for temporal latent vectors is

$$
L_U=\frac{1}{B}\sum_{i=1}^{B}\sum_{t=1}^{T}\left[\log D(H_{i,t})+\log\left(1-D(\widehat H_{i,t})\right)\right].
$$

Here, $D(H_{i,t})$ and $D(\widehat H_{i,t})$ are the discriminator probabilities assigned to observed and generated latent vectors. The discriminator seeks a large $L_U$; the minimax generator seeks a small term for generated data.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset

import RiskLabAI

assert RiskLabAI.__version__ == "3.2.0"

DATA_PATH = ensure_dataset(BOOK_ROOT, "fred_nasdaq")
FIGURES = BOOK_ROOT / "outputs" / "figures" / "03_generative_financial_time_series"
TABLES = BOOK_ROOT / "outputs" / "tables" / "03_generative_financial_time_series"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "03_generative_financial_time_series"
ARTIFACTS = BOOK_ROOT / "outputs" / "artifacts" / "03_generative_financial_time_series"
for directory in [FIGURES, TABLES, FIGURE_DATA, ARTIFACTS]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return "0"
        if number.is_integer():
            return f"{int(number):,}"
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f"{number:.6g}"
        return f"{number:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)

import torch

from RiskLabAI.data.synthetic_data.timegan import TimeGAN, chronological_timegan_data, fit_timegan

payload = np.load(ARTIFACTS / "prepared_timegan_data.npz")
training_windows = payload["training"]
validation_windows = payload["validation"]
minimum = payload["minimum"]
scale = payload["scale"]
feature_names = payload["feature_names"].tolist()
window_length = int(payload["window_length"][0])
assert training_windows.shape == (6409, 20, 2)

In [2]:
objective_model = TimeGAN(2, 4, seed=1)
random_generator = np.random.default_rng(42)
observed = torch.tensor(random_generator.uniform(size=(3, 5, 2)), dtype=torch.float64)
noise = torch.tensor(random_generator.normal(size=(3, 5, 4)), dtype=torch.float64)
losses = objective_model.losses(observed, noise)
with torch.no_grad():
    latent = objective_model.embedding(observed)
    reconstructed = objective_model.recovery(latent).numpy()
    synthetic_latent = objective_model.latent_sequence(noise)
    teacher_latent = objective_model.latent_sequence(noise, latent).numpy()
    real_logits = objective_model.discriminator(latent).numpy()
    fake_logits = objective_model.discriminator(synthetic_latent).numpy()

independent = {
    "reconstruction": sum(np.sum((a - b) ** 2) for a, b in zip(observed.numpy(), reconstructed)) / 3,
    "supervised": sum(np.sum((a - b) ** 2) for a, b in zip(latent.numpy(), teacher_latent)) / 3,
    "discriminator": np.logaddexp(0, -real_logits).sum() / 3 + np.logaddexp(0, fake_logits).sum() / 3,
    "generator_adversarial": -np.logaddexp(0, fake_logits).sum() / 3,
}
objective_labels = {
    "reconstruction": "Reconstruction loss",
    "supervised": "Supervised temporal loss",
    "discriminator": "Discriminator loss",
    "generator_adversarial": "Generator adversarial loss",
}
objective_rows = []
for name in independent:
    library_value = losses[name].item()
    independent_value = independent[name]
    assert np.isclose(library_value, independent_value)
    objective_rows.append({
        "Objective": objective_labels[name],
        "RiskLabAI value": library_value,
        "Independent value": independent_value,
        "Absolute difference": abs(library_value - independent_value),
    })
sum(losses.values()).backward()
assert all(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in objective_model.parameters())
changed_noise = noise.clone()
changed_noise[:, 3:] += 100
assert np.allclose(objective_model.latent_sequence(noise)[:, :3].detach().numpy(), objective_model.latent_sequence(changed_noise)[:, :3].detach().numpy(), atol=0)
objective_table = pd.DataFrame(objective_rows)
objective_table.to_csv(FIGURE_DATA / "objective_check_values.csv", index=False)
save_table(objective_table, "objective_checks", "Independent calculations of the four implemented losses.", "tab:chapter4-objective-checks", fit=True)

,Objective,RiskLabAI value,Independent value,Absolute difference
0,Reconstruction loss,1.019948,1.019948,4.44089e-16
1,Supervised temporal loss,0.111138,0.111138,2.77556e-17
2,Discriminator loss,7.015055,7.015055,1.77636e-15
3,Generator adversarial loss,-3.244339,-3.244339,4.44089e-16


,Objective,RiskLabAI value,Independent value,Absolute difference
0,Reconstruction loss,1.019948,1.019948,4.44089e-16
1,Supervised temporal loss,0.111138,0.111138,2.77556e-17
2,Discriminator loss,7.015055,7.015055,1.77636e-15
3,Generator adversarial loss,-3.244339,-3.244339,4.44089e-16


In [3]:
nodes = pd.DataFrame({
    "Component": ["Observed window", "Embedding", "Latent sequence", "Recovery", "Reconstruction", "Random inputs", "Generator", "Generated latent sequence", "Generated window", "Discriminator"],
    "X": [0, 2, 4, 6, 8, 0, 2, 4, 6, 6],
    "Y": [2, 2, 2, 2, 2, 0, 0, 0, 0, 1],
    "Group": ["data", "map", "latent", "map", "data", "noise", "map", "latent", "data", "judge"],
})
edges = pd.DataFrame({
    "Source": ["Observed window", "Embedding", "Latent sequence", "Recovery", "Random inputs", "Generator", "Generated latent sequence", "Generated latent sequence", "Latent sequence"],
    "Target": ["Embedding", "Latent sequence", "Recovery", "Reconstruction", "Generator", "Generated latent sequence", "Generated window", "Discriminator", "Discriminator"],
})
nodes.to_csv(FIGURE_DATA / "component_nodes.csv", index=False)
edges.to_csv(FIGURE_DATA / "component_edges.csv", index=False)

positions = {row["Component"]: (row["X"], row["Y"]) for _, row in nodes.iterrows()}
colors = {"data": PALETTE["cyan"], "map": PALETTE["amber"], "latent": PALETTE["green"], "noise": PALETTE["violet"], "judge": PALETTE["rose"]}
component_labels = {"Observed window": "Observed\nwindow", "Embedding": "Embedding", "Latent sequence": "Latent\nsequence", "Recovery": "Recovery", "Reconstruction": "Recovered\nwindow", "Random inputs": "Random\ninputs", "Generator": "Generator", "Generated latent sequence": "Generated\nlatent\nsequence", "Generated window": "Generated\nwindow", "Discriminator": "Real or\ngenerated\nclassifier"}
figure, axis = plt.subplots(figsize=(12.0, 4.7), constrained_layout=True)
for _, edge in edges.iterrows():
    start = positions[edge["Source"]]
    end = positions[edge["Target"]]
    axis.annotate("", xy=end, xytext=start, arrowprops={"arrowstyle": "->", "color": PALETTE["text"], "lw": 1.3, "shrinkA": 27, "shrinkB": 27})
for _, row in nodes.iterrows():
    axis.scatter(row["X"], row["Y"], s=2400, color=colors[row["Group"]], edgecolor=PALETTE["text"], linewidth=1.2, zorder=3)
    axis.text(row["X"], row["Y"], component_labels[row["Component"]], ha="center", va="center", color=PALETTE["background"], fontsize=8.3, fontweight="bold", zorder=4)
axis.set_xlim(-0.8, 8.8)
axis.set_ylim(-0.7, 2.7)
axis.set_title("Temporal Generation in a Learned Latent Space")
axis.axis("off")
save_figure(figure, "timegan_component_map")
display(figure)
plt.close(figure)

<Figure size 1200x470 with 1 Axes>

## Three Training Phases

The embedding and recovery parameters solve

$$
\min_{\theta_E,\theta_R}\left(L_R+\lambda L_S\right),
$$

while the generator parameters solve

$$
\min_{\theta_G}\left(L_{G,\mathrm{adv}}+\eta L_S\right),
$$

and the discriminator parameters solve

$$
\min_{\theta_D}(-L_U).
$$

Here, $\theta_E$, $\theta_R$, $\theta_G$, and $\theta_D$ are the adjustable parameters of the embedding, recovery, generator, and discriminator maps. $L_{G,\mathrm{adv}}$ is the term for generated data in the adversarial value; $\lambda\geq0$ weights supervised temporal learning in the embedding step; $\eta\geq0$ weights it in the generator step. The first phase trains reconstruction, the second trains supervised temporal generation, and the third combines all objectives.

In [4]:
settings = {
    "embedding_steps": 20,
    "supervised_steps": 20,
    "joint_steps": 40,
    "batch_size": 64,
    "learning_rate": 0.001,
    "embedding_weight": 1.0,
    "generator_weight": 10.0,
    "seed": 4105,
}
model = TimeGAN(features=2, hidden_size=8, seed=4104)
history = fit_timegan(model, training_windows, **settings)
assert len(history) == 80
assert [entry["phase"] for entry in history] == ["embedding"] * 20 + ["supervised"] * 20 + ["joint"] * 40
assert all(np.isfinite(value) for entry in history for key, value in entry.items() if key != "phase")

history_frame = pd.DataFrame(history).rename(columns={
    "phase": "Phase",
    "reconstruction": "Reconstruction loss",
    "supervised": "Supervised temporal loss",
    "discriminator": "Discriminator loss",
    "generator_adversarial": "Generator adversarial loss",
})
history_frame.insert(0, "Step", np.arange(1, len(history_frame) + 1))
history_frame["Phase"] = history_frame["Phase"].map({"embedding": "Embedding", "supervised": "Supervised", "joint": "Joint"})
history_frame.to_csv(FIGURE_DATA / "training_history.csv", index=False)
torch.save(model.state_dict(), ARTIFACTS / "timegan_state.pt")
(ARTIFACTS / "timegan_settings.json").write_text(json.dumps({"features": 2, "hidden_size": 8, **settings}, indent=2) + "\n", encoding="utf-8")

settings_table = pd.DataFrame({
    "Setting": ["Feature count", "Window length", "Hidden units", "Embedding steps", "Supervised steps", "Joint steps", "Batch size", "Learning rate", "Embedding supervised weight", "Generator supervised weight", "Model seed", "Training seed"],
    "Value": [2, 20, 8, 20, 20, 40, 64, 0.001, 1, 10, 4104, 4105],
})
save_table(settings_table, "timegan_settings", "Fixed settings chosen before the training run with public data.", "tab:chapter4-timegan-settings")

phase_rows = []
for phase in ["Embedding", "Supervised", "Joint"]:
    part = history_frame[history_frame["Phase"] == phase]
    phase_rows.append({
        "Phase": phase,
        "Steps": len(part),
        "First reconstruction loss": part["Reconstruction loss"].iloc[0],
        "Last reconstruction loss": part["Reconstruction loss"].iloc[-1],
        "First supervised loss": part["Supervised temporal loss"].iloc[0],
        "Last supervised loss": part["Supervised temporal loss"].iloc[-1],
    })
phase_summary = pd.DataFrame(phase_rows)
save_table(phase_summary, "training_phase_summary", "Loss values at the beginning and end of each fixed training phase.", "tab:chapter4-training-phases", fit=True)

,Setting,Value
0,Feature count,2
1,Window length,20
2,Hidden units,8
3,Embedding steps,20
4,Supervised steps,20
5,Joint steps,40
6,Batch size,64
7,Learning rate,0.001
8,Embedding supervised weight,1
9,Generator supervised weight,10


,Phase,Steps,First reconstruction loss,Last reconstruction loss,First supervised loss,Last supervised loss
0,Embedding,20,3.838145,2.982736,0.782602,1.064797
1,Supervised,20,3.044207,2.97927,1.116043,0.670759
2,Joint,40,2.962546,1.715758,0.64655,0.10796


,Phase,Steps,First reconstruction loss,Last reconstruction loss,First supervised loss,Last supervised loss
0,Embedding,20,3.838145,2.982736,0.782602,1.064797
1,Supervised,20,3.044207,2.97927,1.116043,0.670759
2,Joint,40,2.962546,1.715758,0.64655,0.10796


In [5]:
loss_columns = ["Reconstruction loss", "Supervised temporal loss", "Discriminator loss", "Generator adversarial loss"]
loss_colors = [PALETTE["cyan"], PALETTE["green"], PALETTE["amber"], PALETTE["rose"]]
figure, axes = plt.subplots(2, 2, figsize=(11.2, 7.6), constrained_layout=True)
for axis, column, color in zip(axes.ravel(), loss_columns, loss_colors):
    axis.plot(history_frame["Step"], history_frame[column], color=color, linewidth=1.8)
    axis.axvline(20.5, color=PALETTE["violet"], linewidth=1.0)
    axis.axvline(40.5, color=PALETTE["violet"], linewidth=1.0)
    axis.set_title(column)
    axis.set_xlabel("Training step")
    axis.set_ylabel("Loss")
    finish_axis(axis)
save_figure(figure, "training_losses")
display(figure)
plt.close(figure)

<Figure size 1120x760 with 4 Axes>

In [6]:
selected_scaled = torch.tensor(training_windows[100:101], dtype=torch.float64)
with torch.no_grad():
    selected_latent = model.embedding(selected_scaled)
    reconstructed_scaled = model.recovery(selected_latent).numpy()[0]
    generated_scaled = model.sample(1, window_length, seed=4108).numpy()[0]
real_window = training_windows[100] * scale + minimum
reconstructed_window = reconstructed_scaled * scale + minimum
generated_window = generated_scaled * scale + minimum

window_rows = []
for source, values in [("Observed training window", real_window), ("Reconstruction", reconstructed_window), ("Generated window", generated_window)]:
    for day in range(window_length):
        for feature_index, feature in enumerate(feature_names):
            window_rows.append({"Source": source, "Relative day": day + 1, "Feature": feature, "Value": values[day, feature_index]})
window_examples = pd.DataFrame(window_rows)
window_examples.to_csv(FIGURE_DATA / "model_window_examples.csv", index=False)

window_summary_rows = []
for source, part in window_examples.groupby("Source", sort=False):
    for feature, values in part.groupby("Feature", sort=False)["Value"]:
        window_summary_rows.append({"Source": source, "Feature": feature, "Mean": values.mean(), "Standard deviation": values.std(ddof=1), "Minimum": values.min(), "Maximum": values.max()})
window_summary = pd.DataFrame(window_summary_rows)
save_table(window_summary, "model_window_summary", "Summary of one fixed observed window, its reconstruction, and one generated window.", "tab:chapter4-model-window-summary", fit=True)

figure, axes = plt.subplots(2, 1, figsize=(10.8, 7.0), constrained_layout=True)
source_styles = [("Observed training window", PALETTE["cyan"], "-"), ("Reconstruction", PALETTE["amber"], "--"), ("Generated window", PALETTE["green"], ":")]
for axis, feature in zip(axes, feature_names):
    for source, color, style in source_styles:
        part = window_examples[(window_examples["Source"] == source) & (window_examples["Feature"] == feature)]
        axis.plot(part["Relative day"], part["Value"], color=color, linestyle=style, linewidth=2.0, marker="o", markersize=3.2, label=source)
    axis.set_ylabel(feature)
    axis.set_title(feature)
    finish_axis(axis)
axes[-1].set_xlabel("Relative day in window")
axes[0].legend(ncol=3, fontsize=9)
save_figure(figure, "reconstruction_generated_windows")
display(figure)
plt.close(figure)

,Source,Feature,Mean,Standard deviation,Minimum,Maximum
0,Observed training window,Daily log return (%),0.111678,0.603073,-1.012456,1.366069
1,Observed training window,Twenty-day return volatility (%),0.791604,0.056651,0.603073,0.845175
2,Reconstruction,Daily log return (%),0.121139,0.062142,0.073487,0.376333
3,Reconstruction,Twenty-day return volatility (%),2.496993,0.072968,2.459964,2.723329
4,Generated window,Daily log return (%),0.113494,0.067934,0.071008,0.388081
5,Generated window,Twenty-day return volatility (%),2.501153,0.072026,2.461897,2.721905


<Figure size 1080x700 with 2 Axes>

## Tests

In [7]:
before = torch.random.get_rng_state().clone()
tiny = TimeGAN(1, 4, seed=8)
assert torch.equal(before, torch.random.get_rng_state())
toy_series = np.sin(np.arange(100) / 4)[:, None]
toy_data = chronological_timegan_data(toy_series, training_end=60, window_length=6)
toy_settings = {"embedding_steps": 2, "supervised_steps": 2, "joint_steps": 2, "batch_size": 5, "seed": 12}
toy_history = fit_timegan(tiny, toy_data["training"], **toy_settings)
twin = TimeGAN(1, 4, seed=8)
assert fit_timegan(twin, toy_data["training"], **toy_settings) == toy_history
assert [entry["phase"] for entry in toy_history] == ["embedding", "embedding", "supervised", "supervised", "joint", "joint"]
assert np.allclose(tiny.sample(20, 6, seed=10).numpy(), twin.sample(20, 6, seed=10).numpy(), atol=0)

training_checks = pd.DataFrame({
    "Check": ["Independent loss calculations", "Finite gradients", "Future noise leaves earlier outputs unchanged", "Three phases in fixed order", "Repeated seeds reproduce training", "Repeated seeds reproduce samples", "Public-data losses finite"],
    "Result": ["Pass", "Pass", "Pass", "Pass", "Pass", "Pass", "Pass"],
})
save_table(training_checks, "training_checks", "Checks for objectives, gradients, temporal order, phases, and deterministic seeds.", "tab:chapter4-training-checks", fit=True)

,Check,Result
0,Independent loss calculations,Pass
1,Finite gradients,Pass
2,Future noise leaves earlier outputs unchanged,Pass
3,Three phases in fixed order,Pass
4,Repeated seeds reproduce training,Pass
5,Repeated seeds reproduce samples,Pass
6,Public-data losses finite,Pass


,Check,Result
0,Independent loss calculations,Pass
1,Finite gradients,Pass
2,Future noise leaves earlier outputs unchanged,Pass
3,Three phases in fixed order,Pass
4,Repeated seeds reproduce training,Pass
5,Repeated seeds reproduce samples,Pass
6,Public-data losses finite,Pass
